# Download CNN/DailyMail articles

This notebook follows the raw-data structure used in `old_preprocessing.ipynb`. It downloads records from the CNN/DailyMail 3.0.0 training split, maps the dataset's `highlights` field to the **`summary`** column, calculates article word counts, retains articles with fewer than 500 words, and saves exactly 300 samples to `data/raw/cnn_dailymail_raw.jsonl`.

## Setup

In [1]:
import json
import os
import random
import re
import time
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd

DATASET_NAME = "abisee/cnn_dailymail"
DATASET_CONFIG = "3.0.0"
DATASET_SPLIT = "train"
DATASET_ROWS_API = "https://datasets-server.huggingface.co/rows"

RAW_DIR = Path("data/raw")
RAW_PATH = RAW_DIR / "cnn_dailymail_raw.jsonl"
SAMPLE_SIZE = 300
MAX_ARTICLE_WORDS_EXCLUSIVE = 500
SEED = 42
SHUFFLE_BUFFER_SIZE = 10_000
OVERWRITE = True

WORD_RE = re.compile(r"\b\w+(?:[’'-]\w+)*\b", flags=re.UNICODE)
RAW_COLUMNS = ["article_id", "article", "summary", "article_word_count"]

RAW_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

print(f"Raw output: {RAW_PATH}")
print(f"Required columns: {RAW_COLUMNS}")
print(f"Target: {SAMPLE_SIZE} articles with < {MAX_ARTICLE_WORDS_EXCLUSIVE} words")

Raw output: data\raw\cnn_dailymail_raw.jsonl
Required columns: ['article_id', 'article', 'summary', 'article_word_count']
Target: 300 articles with < 500 words


## Download and filter CNN/DailyMail

As in `old_preprocessing.ipynb`, each raw record uses `id` as `article_id`, `article` as `article`, and **`highlights` as `summary`**. Pages are visited in a deterministic shuffled order, and collection stops after exactly 300 articles below the 500-word limit are found.

In [2]:
def count_words(text: str) -> int:
    return len(WORD_RE.findall(text))


def fetch_dataset_page(offset: int, length: int = 100, retries: int = 4) -> dict:
    query = urlencode({
        "dataset": DATASET_NAME,
        "config": DATASET_CONFIG,
        "split": DATASET_SPLIT,
        "offset": offset,
        "length": length,
    })
    request_url = f"{DATASET_ROWS_API}?{query}"

    for attempt in range(1, retries + 1):
        try:
            with urlopen(request_url, timeout=120) as response:
                return json.loads(response.read().decode("utf-8"))
        except Exception:
            if attempt == retries:
                raise
            time.sleep(2 ** (attempt - 1))


def iter_dataset_rows():
    # Use the same Hugging Face streaming approach as old_preprocessing.ipynb.
    # The official rows API is a fallback for environments where the Hub
    # metadata connection is unavailable.
    try:
        from datasets import load_dataset

        dataset = load_dataset(
            DATASET_NAME,
            DATASET_CONFIG,
            split=DATASET_SPLIT,
            streaming=True,
        ).shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER_SIZE)
        print("Download method: datasets.load_dataset (streaming)")
        yield from dataset
        return
    except Exception as error:
        print(
            f"datasets.load_dataset unavailable ({type(error).__name__}); "
            "using the official Hugging Face rows API."
        )

    metadata_page = fetch_dataset_page(offset=0, length=1)
    total_rows = metadata_page["num_rows_total"]
    page_size = min(metadata_page["num_rows_per_page"], 100)
    page_offsets = list(range(0, total_rows, page_size))
    random.Random(SEED).shuffle(page_offsets)

    for offset in page_offsets:
        page = fetch_dataset_page(offset=offset, length=page_size)
        for wrapped_row in page["rows"]:
            # Do not use a record if the viewer reports truncated text.
            if not wrapped_row.get("truncated_cells"):
                yield wrapped_row["row"]

records = []
seen_ids = set()
seen_articles = set()
scanned = 0
rejected_at_or_above_500 = 0

for row in iter_dataset_rows():
    scanned += 1
    article_id = (row.get("id") or "").strip()
    article = (row.get("article") or "").strip()
    summary = (row.get("highlights") or "").strip()

    if not article_id or not article or not summary:
        continue
    if article_id in seen_ids or article in seen_articles:
        continue

    article_word_count = count_words(article)
    if article_word_count >= MAX_ARTICLE_WORDS_EXCLUSIVE:
        rejected_at_or_above_500 += 1
        continue

    records.append({
        "article_id": article_id,
        "article": article,
        "summary": summary,
        "article_word_count": article_word_count,
    })
    seen_ids.add(article_id)
    seen_articles.add(article)

    if len(records) == SAMPLE_SIZE:
        break

if len(records) != SAMPLE_SIZE:
    raise RuntimeError(
        f"Found only {len(records)} qualifying articles after scanning {scanned} rows."
    )

print(f"Dataset rows scanned: {scanned:,}")
print(f"Rows rejected at or above 500 words: {rejected_at_or_above_500:,}")
print(f"Qualifying records collected: {len(records)}")

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting GET https://huggingface.co/datasets/abisee/cnn_dailymail/resolve/96df5e686bee6baa90b8bee7c28b81fa3fa6223d/3.0.0/train-00002-of-00003.parquet
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting GET https://huggingface.co/datasets/abisee/cnn_dailymail/resolve/96df5e686bee6baa90b8bee7c28b81fa3fa6223d/3.0.0/train-00001-of-00003.parquet
Retrying in 1s [Retry 1/5].
'[WinError 10038] An operation was attempted on something that is not a socket' thrown while requesting GET https://huggingface.co/datasets/abisee/cnn_dailymail/resolve/96df5e686bee6baa90b8bee7c28b81fa3fa6223d/3.0.0/train-00000-of-00003.parquet
Retrying in 1s [Retry 1/5].
Retrying in 1s [Retry 1/5].


Download method: datasets.load_dataset (streaming)
datasets.load_dataset unavailable (RuntimeError); using the official Hugging Face rows API.
Dataset rows scanned: 972
Rows rejected at or above 500 words: 672
Qualifying records collected: 300


## Save and verify the raw dataset

The output is first written to a temporary file and validated. The existing raw file is replaced only after those checks pass.

In [3]:
if RAW_PATH.exists() and not OVERWRITE:
    raise FileExistsError(f"{RAW_PATH} already exists. Set OVERWRITE=True to replace it.")

temporary_path = RAW_PATH.with_suffix(RAW_PATH.suffix + ".tmp")
try:
    with temporary_path.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    with temporary_path.open("r", encoding="utf-8") as handle:
        saved_records = [json.loads(line) for line in handle if line.strip()]

    assert len(saved_records) == SAMPLE_SIZE
    assert all(set(record) == set(RAW_COLUMNS) for record in saved_records)
    assert all(record["summary"].strip() for record in saved_records)
    assert all(
        count_words(record["article"]) == record["article_word_count"]
        for record in saved_records
    )
    assert all(
        record["article_word_count"] < MAX_ARTICLE_WORDS_EXCLUSIVE
        for record in saved_records
    )

    os.replace(temporary_path, RAW_PATH)
finally:
    if temporary_path.exists():
        temporary_path.unlink()

word_counts = [record["article_word_count"] for record in saved_records]
print(f"Saved {len(saved_records)} records to '{RAW_PATH}'")
print(f"Saved columns: {list(saved_records[0])}")
print(f"Non-empty summaries: {sum(bool(record['summary'].strip()) for record in saved_records)}")
print(f"Word-count range: {min(word_counts)} to {max(word_counts)}")
print("Verification passed.")

Saved 300 records to 'data\raw\cnn_dailymail_raw.jsonl'
Saved columns: ['article_id', 'article', 'summary', 'article_word_count']
Non-empty summaries: 300
Word-count range: 96 to 499
Verification passed.
